<a href="https://colab.research.google.com/github/maybraniswe/Retail-Store-Sales-Analysis-and-Prediction/blob/main/PDSE%20Project_Retail%20Store%20Sales%20Analysis%20and%20Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retail Store Sales Analysis and Prediction

In [3]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler

##Data Loading

In [15]:
# read the data from csv file
df = pd.read_csv('https://raw.githubusercontent.com/maybraniswe/Retail-Store-Sales-Analysis-and-Prediction/refs/heads/main/retail_store_sales.csv')

# Dataset size check (rows, columns)
print (df.shape)

(12575, 11)


In [13]:
# View data the first 5 rows
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,item_10_pat,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,1
1,TXN_3731986,CUST_22,Milk Products,item_17_milk,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,1
2,TXN_9303719,CUST_02,Butchers,item_12_but,21.5,2.0,43.0,Credit Card,Online,2022-10-05,0
3,TXN_9458126,CUST_06,Beverages,item_16_bev,27.5,9.0,247.5,Credit Card,Online,2022-05-07,0
4,TXN_4575373,CUST_05,Food,item_6_food,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,0


In [5]:
# Check summary statistics
df.describe()

,Price Per Unit,Quantity,Total Spent
count,11966.000000,11971.000000,11971.000000
mean,23.365912,5.536380,129.652577
std,10.743519,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [16]:
# Check summary statistics
print (df.describe(include='all'))

       Transaction ID Customer ID                       Category        Item  \
count           12575       12575                          12575       11362   
unique          12575          25                              8         200   
top       TXN_2407494     CUST_05  Electric household essentials  Item_2_BEV   
freq                1         544                           1591         126   
mean              NaN         NaN                            NaN         NaN   
std               NaN         NaN                            NaN         NaN   
min               NaN         NaN                            NaN         NaN   
25%               NaN         NaN                            NaN         NaN   
50%               NaN         NaN                            NaN         NaN   
75%               NaN         NaN                            NaN         NaN   
max               NaN         NaN                            NaN         NaN   

        Price Per Unit      Quantity   

In [6]:
# Check structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [7]:
# Checking for missing values
df.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


## Missing Values

In [8]:
# Fill missing numeric values using median (for outliers)
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Price Per Unit'].median())
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# Recalculate Total Spent
df['Total Spent'] = df['Price Per Unit'] * df['Quantity']

# Fill Discount Applied with 0 (assume no discount)
df['Discount Applied'] = df['Discount Applied'].fillna(0)

# Fill Item using group based method
df['Item'] = df.groupby('Price Per Unit')['Item'].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'unknown')
)
print (df.isnull().sum())


Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


##Converting Data Types

In [10]:
# Convert numeric columns
num_cols = ['Price Per Unit', 'Quantity', 'Total Spent', 'Discount Applied']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert date column
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# Convert categorical columns
cat_cols = ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Payment Method', 'Location']
for col in cat_cols:
    df[col] = df[col].astype('category')

print(df.dtypes)

Transaction ID            category
Customer ID               category
Category                  category
Item                      category
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[ns]
Discount Applied             int64
dtype: object


##Standardizing Inconsistent Data

In [11]:
# Clean text data (remove spaces, lowercase)
df['Item'] = df['Item'].str.strip().str.lower()
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,item_10_pat,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,1
1,TXN_3731986,CUST_22,Milk Products,item_17_milk,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,1
2,TXN_9303719,CUST_02,Butchers,item_12_but,21.5,2.0,43.0,Credit Card,Online,2022-10-05,0
3,TXN_9458126,CUST_06,Beverages,item_16_bev,27.5,9.0,247.5,Credit Card,Online,2022-05-07,0
4,TXN_4575373,CUST_05,Food,item_6_food,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,0


##Handling Duplicates

In [12]:
# Remove duplicate rows
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after:", df.duplicated().sum())

Duplicate rows before: 0
Duplicate rows after: 0
